# ML-04 - Refresh Opportunity Data Contract

**Lane:** Refresh / Content Opportunity Scoring  
**Decision moment:** the end of 2026-03-31  
**Goal:** rank visible pages for human review before the next month is observed.

## 1. Contract in plain words

1. **What one row means:** one pseudonymized content page belonging to one pseudonymized client. Daily rows are aggregated to that page-month grain before features are built.
2. **Tables:** `fact_content_daily_performance/month=2026-03` supplies the features; the 2026-04 partition supplies only the later outcome. I do not use the June `_sample` table.
3. **Time window:** features use 2026-03-01 through 2026-03-31. The label uses 2026-04-01 through 2026-04-30. The decision happens after March closes and before April is observed.
4. **What I predict/rank:** `future_decline_label = 1` when April GSC impressions are below 80% of March impressions. The model score is decision support for a refresh-review queue, not proof that refreshing causes recovery.
5. **What I deliberately exclude:** April values, the copied label used in the teaching trap, GA4 fields, client/content IDs as model inputs, private URLs/queries, product flags, and the final June sample. They are unavailable at the decision moment, out of scope, identifiers/private, label-derived, or reserved for later testing.

**Eligibility:** at least 20 days with `gsc_data_available IS TRUE` in both months and at least 100 March impressions. This avoids treating unavailable days as observed zero traffic.

In [1]:
from pathlib import Path
import os

import duckdb
import numpy as np
import pandas as pd
from IPython.display import display
from huggingface_hub import HfFolder, hf_hub_download

REPO = "FlyRank/internship-warehouse"
TOKEN = os.getenv("HF_TOKEN") or HfFolder.get_token()
if not TOKEN:
    raise RuntimeError(
        "No Hugging Face token found. Add HF_TOKEN as a Colab Secret or log in locally."
    )

def warehouse_file(month):
    filename = f"fact_content_daily_performance/month={month}/data_0.parquet"
    path = hf_hub_download(REPO, repo_type="dataset", filename=filename, token=TOKEN)
    return Path(path).as_posix()

MARCH_PATH = warehouse_file("2026-03")
APRIL_PATH = warehouse_file("2026-04")
con = duckdb.connect()

def run_query(sql):
    return con.execute(sql).df()

print("Warehouse partitions ready: 2026-03 features -> 2026-04 outcome")

D:\New folder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Warehouse partitions ready: 2026-03 features -> 2026-04 outcome


## 2. Fields and the five-feature limit

### Features - exactly five

1. `log_impressions` - March impression volume on a compressed scale. **Knowable at the decision moment because** all March GSC days have closed.
2. `ctr_pct` - March clicks divided by March impressions, multiplied by 100. **Knowable at the decision moment because** both totals are observed by March 31.
3. `avg_position` - March impression-weighted average position. **Knowable at the decision moment because** it uses only March `gsc_sum_position` and impressions.
4. `active_days` - March days with at least one impression. **Knowable at the decision moment because** every counted day is on or before March 31.
5. `late_vs_early_change` - change from March 1-16 to March 17-31, divided by at least one early impression. **Knowable at the decision moment because** both halves end before scoring.

### Other field classes

- **Label/proxy:** `future_decline_label`, computed only from April versus March impressions. It is never retained as a feature.
- **Context:** `client_hash_id` and `content_hash_id` are used for joins, client-held-out splitting, and reading results only.
- **Excluded:** `april_impressions`, URLs, queries, GA4 metrics, warehouse/product flags, June sample values, and any label-derived column.

## 3. Three verification queries

These are the contract checks: grain, slice size/date span, and explicit availability.

In [2]:
# Verification query 1 of 3: the declared daily source grain must have no duplicates.
query_1_grain = f"""
SELECT
    report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_at_grain
FROM read_parquet('{MARCH_PATH}')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 10
"""
grain_violations = run_query(query_1_grain)
print("Grain violations (expected 0 rows):")
display(grain_violations)

Grain violations (expected 0 rows):


,report_date,client_hash_id,content_hash_id,rows_at_grain


In [3]:
# Verification query 2 of 3: row count, page count, clients, and date span.
query_2_slice = f"""
SELECT
    COUNT(*) AS source_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id)) AS distinct_pages,
    COUNT(DISTINCT client_hash_id) AS clients,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM read_parquet('{MARCH_PATH}')
"""
slice_check = run_query(query_2_slice)
display(slice_check)

,source_rows,distinct_pages,clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [4]:
# Verification query 3 of 3: IS TRUE distinguishes measured rows from zero-filled rows.
query_3_availability = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE
    ) AS both_available_rows,
    COUNT(DISTINCT (client_hash_id, content_hash_id)) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_pages
FROM read_parquet('{MARCH_PATH}')
"""
availability_check = run_query(query_3_availability)
display(availability_check)

,total_rows,gsc_available_rows,ga4_available_rows,both_available_rows,gsc_available_pages
0,9841378,3611061,413966,364347,176738


## 4. Five-feature frame

The aggregation below uses only rows explicitly marked as GSC-available. April is joined only to create the later label and is not included in `FEATURES`.

In [5]:
feature_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_gsc_days,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        SUM(gsc_sum_position) AS march_sum_position,
        SUM(CASE WHEN gsc_impressions > 0 THEN 1 ELSE 0 END) AS active_days,
        SUM(CASE WHEN report_date < DATE '2026-03-17'
                 THEN gsc_impressions ELSE 0 END) AS early_impressions,
        SUM(CASE WHEN report_date >= DATE '2026-03-17'
                 THEN gsc_impressions ELSE 0 END) AS late_impressions
    FROM read_parquet('{MARCH_PATH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
),
april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_gsc_days,
        SUM(gsc_impressions) AS april_impressions
    FROM read_parquet('{APRIL_PATH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT
    m.client_hash_id,
    m.content_hash_id,
    LN(1 + m.march_impressions) AS log_impressions,
    100.0 * m.march_clicks / NULLIF(m.march_impressions, 0) AS ctr_pct,
    m.march_sum_position / NULLIF(m.march_impressions, 0) AS avg_position,
    m.active_days,
    (m.late_impressions - m.early_impressions)
        / GREATEST(m.early_impressions, 1.0) AS late_vs_early_change,
    m.march_impressions,
    a.april_impressions,
    CAST(a.april_impressions < 0.8 * m.march_impressions AS INTEGER)
        AS future_decline_label
FROM march AS m
INNER JOIN april AS a USING (client_hash_id, content_hash_id)
WHERE m.march_gsc_days >= 20
  AND a.april_gsc_days >= 20
  AND m.march_impressions >= 100
"""

feature_frame = run_query(feature_sql)
FEATURES = [
    "log_impressions",
    "ctr_pct",
    "avg_position",
    "active_days",
    "late_vs_early_change",
]
assert len(FEATURES) == 5
print(f"Eligible page rows: {len(feature_frame):,}")
print(f"Clients: {feature_frame['client_hash_id'].nunique()}")
print(f"Observed decline-label rate: {feature_frame['future_decline_label'].mean():.1%}")
display(feature_frame[FEATURES + ["future_decline_label"]].head(8))

Eligible page rows: 88,941
Clients: 37
Observed decline-label rate: 51.3%


,log_impressions,ctr_pct,avg_position,active_days,late_vs_early_change,future_decline_label
0,7.546974,0.000000,4.806230,29.0,-0.385337,0
1,8.541495,0.370949,4.703241,29.0,0.492457,1
2,7.020191,0.268336,16.385510,29.0,-0.065744,0
3,8.783856,0.015321,42.953884,29.0,0.271051,0
4,6.685861,0.000000,17.361250,29.0,0.077922,1
5,8.305978,0.024710,45.118359,29.0,0.335257,0
6,6.182085,0.000000,17.869565,29.0,0.072961,0
7,7.905442,0.110660,10.162302,29.0,1.534550,0


## 5. Deliberate leakage trap

I first score the five honest features using a client-held-out split. Then I intentionally copy the future label into a feature, score again, and delete that column. A perfect result here is evidence of leakage, not model quality.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline

y = feature_frame["future_decline_label"].astype(int)
groups = feature_frame["client_hash_id"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(feature_frame, y, groups))

def held_out_auc(columns):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=150,
            max_depth=8,
            min_samples_leaf=20,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        ),
    )
    model.fit(feature_frame.iloc[train_idx][columns], y.iloc[train_idx])
    probability = model.predict_proba(feature_frame.iloc[test_idx][columns])[:, 1]
    return roc_auc_score(y.iloc[test_idx], probability)

honest_auc = held_out_auc(FEATURES)

# Add the forbidden future answer on purpose.
feature_frame["future_decline_label_copy"] = y
leaked_auc = held_out_auc(FEATURES + ["future_decline_label_copy"])

# Delete the trap and prove the retained frame is honest again.
feature_frame.drop(columns="future_decline_label_copy", inplace=True)
assert "future_decline_label_copy" not in feature_frame.columns
assert set(FEATURES).isdisjoint({"future_decline_label", "april_impressions"})

print(f"Train: {len(train_idx):,} rows / {groups.iloc[train_idx].nunique()} clients")
print(f"Test:  {len(test_idx):,} rows / {groups.iloc[test_idx].nunique()} clients")
print(f"Honest five-feature ROC AUC: {honest_auc:.3f}")
print(f"With copied future label:      {leaked_auc:.3f}")
print("Copied label deleted; honest retained feature list:", FEATURES)

Train: 76,603 rows / 29 clients
Test:  12,338 rows / 8 clients
Honest five-feature ROC AUC: 0.706
With copied future label:      1.000
Copied label deleted; honest retained feature list: ['log_impressions', 'ctr_pct', 'avg_position', 'active_days', 'late_vs_early_change']


## 6. Named limitation

**Unbalanced client history and selective availability:** only 3,611,061 of 9,841,378 March rows have GSC data explicitly available, and eligible pages come from clients with enough March and April coverage. The observed 0.706 held-out AUC is therefore directional decision-support for this eligible slice; it does not establish causal refresh impact and should not be generalized to clients or pages with missing history.

## Self-check

- [x] Five plain-words contract answers are present.
- [x] Exactly three verification queries are executed with visible outputs.
- [x] Availability is filtered with `IS TRUE`.
- [x] The frame contains exactly five model features with an availability line for each.
- [x] The deliberate leaked column is scored, deleted, and excluded from the retained list.
- [x] One slice limitation is named with careful, non-causal framing.
- [x] No client names, URLs, private queries, credentials, or final-month sample inputs appear.